In [1]:
print("Ok!")

Ok!


In [2]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

from langchain_core.prompts import PromptTemplate
from langchain_community.llms import CTransformers

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from pinecone import Pinecone, ServerlessSpec
import os


c:\Users\HP\anaconda3\envs\mragchatbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Pinecone Setup
# -----------------------------
os.environ["PINECONE_API_KEY"] = "pcsk_2SrqWF_TwCsRJJ8ALB37vTgan1GhMuXv2gmJPwW3mYDGDivkvGCcvAr7JfH9Xm369rWWgm"
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "medical-chatbot"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


In [4]:
# Load PDF & Chunk

def load_pdf(data_dir):
    loader = DirectoryLoader(data_dir, glob="*.pdf", loader_cls=PyPDFLoader)
    return loader.load()

extracted_data = load_pdf("data/")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
text_chunks = text_splitter.split_documents(extracted_data)

In [5]:
# Embeddings & Vector Store
# -----------------------------
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

docsearch = PineconeVectorStore.from_texts(
    [t.page_content for t in text_chunks],
    embedding=embeddings,
    index_name=index_name
)

C:\Users\HP\AppData\Local\Temp\ipykernel_3220\774205904.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [6]:
# LLM (CTransformers)
# -----------------------------
llm = CTransformers(
    model="model/llama-2-7b-chat.ggmlv3.q4_0.bin",
    model_type="llama",
    config={'max_new_tokens': 512, 'temperature': 0.8}
)

In [7]:
# Prompt Template

prompt_template = """
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say you don't know.

Context:
{context}

Question:
{question}

Helpful answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

In [9]:
# RAG PIPELINE (NO CHAINS)
# -----------------------------
retriever = docsearch.as_retriever(search_kwargs={"k": 2})

rag_pipeline = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [10]:
# Query
# -----------------------------
query = "What are the symptoms of diabetes?"
response = rag_pipeline.invoke(query)

print("Answer:", response)

Number of tokens (513) exceeded maximum context length (512).
Number of tokens (514) exceeded maximum context length (512).
Number of tokens (515) exceeded maximum context length (512).
Number of tokens (516) exceeded maximum context length (512).
Number of tokens (517) exceeded maximum context length (512).
Number of tokens (518) exceeded maximum context length (512).
Number of tokens (519) exceeded maximum context length (512).
Number of tokens (520) exceeded maximum context length (512).
Number of tokens (521) exceeded maximum context length (512).
Number of tokens (522) exceeded maximum context length (512).
Number of tokens (523) exceeded maximum context length (512).
Number of tokens (524) exceeded maximum context length (512).
Number of tokens (525) exceeded maximum context length (512).
Number of tokens (526) exceeded maximum context length (512).
Number of tokens (527) exceeded maximum context length (512).
Number of tokens (528) exceeded maximum context length (512).
Number o

Answer: The symptoms of diabetes can vary from person to person, but some common early signs include lethargy, extreme thirst, frequent urination, sudden weight loss, slow wound healing, urinary tract infections, gum disease, or blurred vision. It is not unusual for Type II diabetes to be detected while a patient is seeing a doctor about another health concern that is actually being caused by the yet undiagnosed diabetesdiagnosed diabetesedgnosed diabetes diabetesedgosed diabetesedgnosed diabetes.
